# Turtle Breakout 03 — intraday windows and volatility-engine sizing

Pre-registration: `discussions/Turtle_Breakout/005_claude_tb03_preregistration.md`. Code: `turtle_breakout.py` (h4 periods, engine sizing), `turtle_intraday.py`.

Nine strategies on BTC/ETH/SOL/XRP perpetuals (long only, 2% HWM risk, fees + slippage + funding):

| timeframe | entry / trailing stop / ATR |
|---|---|
| `daily` | 20-day high / 10-day low / 20 days (TB 01) |
| `h4_candles` | 20 × 4h bars (80 h) / 10 bars (40 h) / 20 bars |
| `last_4h` | high of the last 4 hours / low of the last 2 hours / 4 hours (1h bars) |

× sizing `atr` (original), `engine` (engine sets the size), `engine_stop` (engine sets size and fixed stop). Engine unit $U=c\,\sigma_{\text{engine}}P$, with $c$ a causal median of ATR/$(\sigma P)$.

Per strategy: G1 random entries, G2 vs TB 02 benchmarks (Bonferroni over 9). Contrasts C1–C8 (Bonferroni over 8).

Environment: `TB01_DATA_DIR`, `TB03_SYNTHETIC=1`, `TB03_NULL_SCALE` (e.g. 0.1 for a quick look; the decision uses 1).

In [ ]:
import os, sys, json, time, subprocess, tempfile, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dotenv import load_dotenv
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'data_engineering').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src')); load_dotenv(ROOT / '.env')
from data_engineering.modeling import turtle_breakout as tb, turtle_benchmarks as bm, turtle_intraday as ti, vwap_drift as vd

SYNTHETIC = os.getenv('TB03_SYNTHETIC', '0') == '1'
SCALE = float(os.getenv('TB03_NULL_SCALE', '1'))
ASSETS = list(tb.ASSETS); CP = ti.CParams()
if SYNTHETIC:
    OUT = Path(tempfile.mkdtemp(prefix='tb03-')) / '03'; CP = ti.CParams(n_boot=300); SCALE = min(SCALE, 0.05)
else:
    OUT = ROOT / 'notebooks' / 'Turtle_Breakout' / '03'
    base = Path(os.environ['BINANCE_UM_DIR']) if os.environ.get('BINANCE_UM_DIR') else Path(os.environ['DERIBIT_DATA_ROOT']).parent / 'Binance'
    DATA = Path(os.environ.get('TB01_DATA_DIR', str(base / 'long')))
OUT.mkdir(parents=True, exist_ok=True)
env = os.environ.copy(); env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
for t in ('test_turtle_breakout.py', 'test_turtle_benchmarks.py', 'test_turtle_intraday.py'):
    chk = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', t], cwd=str(ROOT), env=env, capture_output=True, text=True)
    print(t, chk.stderr.strip().splitlines()[-1]); assert chk.returncode == 0, t
FITS = vd.load_engine(ROOT / 'notebooks' / 'Volatility_Dynamics' / '18' / '18_manifest.json')
print(json.dumps(ti.registration(CP), indent=1, default=str)[:3000])

## 1. Data

In [ ]:
T0 = time.time(); B5, FUND = {}, {}
for k, a in enumerate(ASSETS):
    if SYNTHETIC:
        b = tb.synthetic_5m(2200, seed=40 + k, trend=0.01, start='2019-11-01'); f = tb.synthetic_funding(b, 0.0001)
    else:
        b = tb.bars_5m(pd.read_parquet(DATA / f'{a}USDT_um_1m.parquet', columns=['open_time', 'open', 'high', 'low', 'close']))
        f = tb.funding_events(pd.read_parquet(DATA / 'basis' / 'funding' / f'{a}USDT_funding.parquet'))
    B5[a], FUND[a] = b, f; print(a, len(b), f'{time.time() - T0:.0f}s')
END = min(b.index[-1] for b in B5.values())
print('window', tb.COMMON_START.date(), '->', END)

## 2. Nine strategies: simulation, random-entry nulls, benchmark tests

In [ ]:
RES, ROWS, BTEST, NULLS = {}, [], [], []
for tf in ti.TIMEFRAMES:
    p0 = ti.strategy_params(tf, 'engine')
    t1 = time.time(); DATA_TF = {a: tb.prepare_asset(a, B5[a], FUND[a], p0, engine=FITS) for a in ASSETS}
    Cp, Fp = bm.daily_panel(DATA_TF, tb.COMMON_START, END)
    print(f'{tf}: prepared in {time.time() - t1:.0f}s')
    for sz in ti.SIZINGS:
        name = f'{tf}|{sz}'; p = ti.strategy_params(tf, sz); t1 = time.time()
        E, T, ST = tb.simulate(DATA_TF, p, start=tb.COMMON_START, end=END)
        n_null = max(20, int(round(ti.N_NULL[tf] * SCALE)))
        NULL = tb.random_portfolios(DATA_TF, p, ST['p_entry'], tb.COMMON_START, END, n=n_null, seed=p.seed + 2)
        row, tests = ti.evaluate_strategy(name, E, T, ST, NULL, Cp, Fp, p.fee + p.slip, CP)
        RES[name] = (E, T, ST); ROWS.append(row); NULLS.append(NULL.assign(strategy=name))
        BTEST += [{'strategy': name, 'benchmark': k, **v} for k, v in tests.items()]
        print(f"  {name:24s} cagr {row['cagr']:7.3f} sharpe {row['sharpe']:5.2f} maxdd {row['max_dd']:6.3f} trades {row['trades']:5d} "
              f"lev {row['mean_leverage']:4.2f} null share {row['null_share_cagr_ge_real']:.4f} -> {row['verdict']}  ({time.time() - t1:.0f}s)")
S = pd.DataFrame(ROWS).set_index('strategy')
display(S[['cagr', 'sharpe', 'max_dd', 'trades', 'win_rate', 'profit_factor', 'avg_hold_days', 'mean_leverage', 'leverage_capped_entries',
           'engine_used', 'engine_fallback', 'fees', 'funding', 'null_cagr_median', 'null_share_cagr_ge_real', 'G1',
           'B1_diff', 'B1_lo', 'B2_lo', 'B3_lo', 'G2', 'verdict']].round(3))

## 3. Contrasts (Bonferroni over 8)

In [ ]:
EQ = {k: v[0] for k, v in RES.items()}
CT = ti.contrasts(EQ, CP)
display(CT.round(3))

## 4. Reported: per-asset P&L, years, benchmark detail

In [ ]:
per_asset = pd.concat({k: v[1].groupby('asset').agg(trades=('pnl', 'size'), pnl=('pnl', 'sum'), fees=('fees', 'sum'),
                        funding=('funding', 'sum'), ret_notional_sum=('ret_notional', 'sum')) for k, v in RES.items()}, names=['strategy'])
display(per_asset.round(2))
YR = pd.DataFrame({k: tb.yearly(v[0]) for k, v in RES.items()})
display((100 * YR).round(1))
BT = pd.DataFrame(BTEST); display(BT.round(3))

## 5. Figures

In [ ]:
Cd, Fd = bm.daily_panel({a: tb.prepare_asset(a, B5[a], FUND[a], tb.Params()) for a in ASSETS}, tb.COMMON_START, END)
B1 = bm.constant_mix(Cd, Fd, min(S.loc['daily|atr', 'mean_leverage'], 1.0), 'M', True, tb.Params().fee + tb.Params().slip)
cols = {'atr': '#1f5aa6', 'engine': '#c2571a', 'engine_stop': '#3a7d44'}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), sharey=False)
for ax, tf in zip(axes, ti.TIMEFRAMES):
    for sz, c in cols.items():
        e = tb.daily(RES[f'{tf}|{sz}'][0]); ax.plot(e.index, e.clip(lower=1), color=c, lw=1.3, label=sz)
    b1 = tb.daily(B1); ax.plot(b1.index, b1, color='0.5', lw=1, ls='--', label='B1 perp monthly')
    ax.set_yscale('log'); ax.set_title(tf, loc='left', fontsize=10); ax.grid(alpha=0.25)
axes[0].set_ylabel('equity, USD (log)'); axes[0].legend(frameon=False, fontsize=8)
fig.suptitle('Turtle breakout: timeframe x sizing, net of fees, slippage and funding', fontsize=11)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
y = np.arange(len(CT))
ax.errorbar(CT['diff'], y, xerr=[CT['diff'] - CT.lo, CT.hi - CT['diff']], fmt='o', color='#1f5aa6', capsize=3)
ax.axvline(0, color='k', lw=0.8); ax.set_yticks(y); ax.set_yticklabels([f"{r.contrast}: {r.a} − {r.b}" for r in CT.itertuples()], fontsize=8)
ax.invert_yaxis(); ax.set_xlabel('Sharpe difference (Bonferroni 1 − 0.05/8 interval)'); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## 6. Exports

In [ ]:
S.to_csv(OUT / '03_strategies.csv', float_format='%.6g'); CT.to_csv(OUT / '03_contrasts.csv', index=False, float_format='%.6g')
BT.to_csv(OUT / '03_benchmark_tests.csv', index=False, float_format='%.6g'); YR.to_csv(OUT / '03_yearly.csv', float_format='%.6g')
per_asset.to_csv(OUT / '03_per_asset.csv', float_format='%.6g')
pd.DataFrame({k: tb.daily(v[0]) for k, v in RES.items()} | {'B1_perp_monthly': tb.daily(B1)}).to_csv(OUT / '03_equity_daily.csv', float_format='%.2f')
pd.concat([v[1].assign(strategy=k) for k, v in RES.items()], ignore_index=True).to_csv(OUT / '03_trades.csv.gz', index=False, float_format='%.6g')
pd.concat(NULLS, ignore_index=True).to_csv(OUT / '03_null_runs.csv.gz', index=False, float_format='%.5g')
(OUT / '03_decision.json').write_text(json.dumps({'strategies': S[['cagr', 'sharpe', 'max_dd', 'G1', 'G2', 'verdict']].to_dict('index'),
    'contrasts': CT[['contrast', 'a', 'b', 'diff', 'lo', 'hi', 'result']].to_dict('records'),
    'null_scale': SCALE, 'registration': ti.registration(CP)}, indent=1, default=float))
(OUT / '03_manifest.json').write_text(json.dumps({'module': 'Turtle_Breakout_03', 'version': ti.VERSION, 'synthetic': SYNTHETIC,
    'null_scale': SCALE, 'window': [str(tb.COMMON_START), str(END)], 'runtime_s': round(time.time() - T0, 1)}, indent=1))
print(sorted(p.name for p in OUT.iterdir()))